In [94]:
import pandas as pd
import matplotlib.pyplot as plt

In [95]:
df = pd.read_csv('../data/processed/MDM_Population.csv')

In [96]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 163364 entries, 0 to 163363
Data columns (total 90 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   PATID                         163364 non-null  object 
 1   FirstNM                       162862 non-null  object 
 2   LastNM                        163009 non-null  object 
 3   MiddleNM                      31759 non-null   object 
 4   SuffixNM                      359 non-null     object 
 5   BirthDT                       163012 non-null  object 
 6   SSN                           58268 non-null   object 
 7   AddressLine1                  158149 non-null  object 
 8   AddressLine2                  48229 non-null   object 
 9   CityNM                        159240 non-null  object 
 10  ZipCD                         158899 non-null  object 
 11  StateCD                       158818 non-null  object 
 12  CountryNM                     161699 non-nul

In [97]:
import pandas as pd
import numpy as np

# =========================================================
# EMPI DETERMINISTIC MATCHING (SAFE BLOCKING VERSION)
# =========================================================

df_work = df.copy()

# ---------------------------------------------------------
# STANDARDIZATION
# ---------------------------------------------------------

str_cols = [
    "FirstNM_clean",
    "LastNM_clean",
    "BirthDT_clean",
    "AddressLine1_clean",
    "Email_clean",
    "SexAtBirthDSC_clean"
]

for c in str_cols:
    if c in df_work.columns:
        df_work[c] = (
            df_work[c]
            .replace(["", " ", "nan", "None", "NULL"], np.nan)
            .astype("string")
            .str.upper()
            .str.strip()
        )

num_cols = [
    "BirthYear",
    "SSN_clean",
    "SSN_Last4",
    "ZipCD_base",
    "PhonePrimary_clean"
]

for c in num_cols:
    if c in df_work.columns:
        df_work[c] = pd.to_numeric(df_work[c], errors="coerce")

# ---------------------------------------------------------
# BASIC FILTER
# ---------------------------------------------------------

df_work = df_work[
    df_work["LastNM_clean"].notna() &
    df_work["BirthYear"].notna()
].copy()

# =========================================================
# SAFE BLOCK BUILDER (KEY FIX)
# =========================================================

def safe_block(df, col, max_block_size=3000, min_non_null_ratio=0.3):
    """
    Prevents:
    - huge blocks (cartesian explosion)
    - junk keys (NULL-heavy fields)
    """

    tmp = df.copy()

    # remove null keys
    tmp = tmp[tmp[col].notna()]

    # drop low-quality columns (too many missing values)
    non_null_ratio = tmp[col].notna().mean()
    if non_null_ratio < min_non_null_ratio:
        return pd.DataFrame()  # skip block entirely

    # cap block sizes
    counts = tmp[col].value_counts()
    valid_keys = counts[counts <= max_block_size].index
    tmp = tmp[tmp[col].isin(valid_keys)]

    return tmp

# =========================================================
# BLOCK KEYS (same as yours but safer usage)
# =========================================================

df_work["B1"] = df_work["SSN_clean"]

df_work["B3"] = df_work["LastNM_clean"] + "_" + df_work["BirthDT_clean"].astype("string")

df_work["B4"] = (
    df_work["LastNM_clean"] + "_" +
    df_work["BirthYear"].astype(int).astype(str) + "_" +
    df_work["FirstNM_clean"].fillna("").str[:3]
)

df_work["B5"] = df_work["PhonePrimary_clean"]

df_work["B6"] = df_work["Email_clean"]

df_work["B7"] = (
    df_work["LastNM_clean"] + "_" +
    df_work["ZipCD_base"].astype("string") + "_" +
    df_work["BirthYear"].astype(int).astype(str)
)

df_work["B8"] = (
    df_work["FirstNM_clean"].fillna("").str[:1] + "_" +
    df_work["LastNM_clean"].fillna("").str[:1] + "_" +
    df_work["BirthYear"].astype(int).astype(str)
)

df_work["B9"] = (
    df_work["LastNM_clean"] + "_" +
    df_work["FirstNM_clean"] + "_" +
    df_work["SSN_Last4"].astype("string")
)

# =========================================================
# BLOCK PAIR BUILDER (SAFE)
# =========================================================

def build_pairs(df, col, block_name):
    tmp = safe_block(df, col)

    if tmp.empty:
        return pd.DataFrame()

    pairs = tmp.merge(tmp, on=col, suffixes=("_L", "_R"))
    pairs = pairs[pairs["PATID_L"] < pairs["PATID_R"]]

    pairs["BLOCK"] = block_name
    return pairs

# =========================================================
# BUILD ALL BLOCKS (SAFE UNION)
# =========================================================

block_dfs = []

for col, name in [
    ("B1", "B1_SSN"),
    ("B3", "B3_LAST_DOB"),
    ("B4", "B4_LAST_BYR_FN3"),
    ("B5", "B5_PHONE"),
    ("B6", "B6_EMAIL"),
    ("B7", "B7_LAST_ZIP_BYR"),
    ("B8", "B8_INIT_INIT_BYR"),
    ("B9", "B9_LAST_FIRST_SSN4"),
]:
    b = build_pairs(df_work, col, name)
    if not b.empty:
        block_dfs.append(b)

pairs = pd.concat(block_dfs, ignore_index=True)

# =========================================================
# FINAL CLEANING
# =========================================================

pairs = pairs[
    ~(
        pairs["FirstNM_clean_L"].isna() &
        pairs["LastNM_clean_L"].isna() &
        pairs["BirthDT_clean_L"].isna() &
        pairs["SSN_clean_L"].isna()
    )
]

# =========================================================
# RULES (UNCHANGED LOGIC)
# =========================================================

rules = []

# ---------------------------------------------------------
# RULE 1: SSN EXACT
# ---------------------------------------------------------

r1 = pairs[
    pairs["SSN_clean_L"].notna() &
    (pairs["SSN_clean_L"] == pairs["SSN_clean_R"])
].copy()

r1["MATCH_RULE"] = "EXACT_SSN"
r1["MATCH_CONFIDENCE"] = 1.00
rules.append(r1)


# ---------------------------------------------------------
# RULE 2B: NAME + DOB + PHONE
# ---------------------------------------------------------

r2b = pairs[
    (pairs["FirstNM_clean_L"] == pairs["FirstNM_clean_R"]) &
    (pairs["LastNM_clean_L"] == pairs["LastNM_clean_R"]) &
    (pairs["BirthDT_clean_L"] == pairs["BirthDT_clean_R"]) &
    (pairs["PhonePrimary_clean_L"] == pairs["PhonePrimary_clean_R"])
].copy()

r2b["MATCH_RULE"] = "NAME_DOB_PHONE"
r2b["MATCH_CONFIDENCE"] = 0.985
rules.append(r2b)

# ---------------------------------------------------------
# RULE 2C: NAME + DOB + EMAIL
# ---------------------------------------------------------

r2c = pairs[
    (pairs["FirstNM_clean_L"] == pairs["FirstNM_clean_R"]) &
    (pairs["LastNM_clean_L"] == pairs["LastNM_clean_R"]) &
    (pairs["BirthDT_clean_L"] == pairs["BirthDT_clean_R"]) &
    (pairs["Email_clean_L"] == pairs["Email_clean_R"])
].copy()

r2c["MATCH_RULE"] = "NAME_DOB_EMAIL"
r2c["MATCH_CONFIDENCE"] = 0.99
rules.append(r2c)

# ---------------------------------------------------------
# RULE 2D: NAME + DOB + ADDRESS
# ---------------------------------------------------------

r2d = pairs[
    (pairs["FirstNM_clean_L"] == pairs["FirstNM_clean_R"]) &
    (pairs["LastNM_clean_L"] == pairs["LastNM_clean_R"]) &
    (pairs["BirthDT_clean_L"] == pairs["BirthDT_clean_R"]) &
    (pairs["AddressLine1_clean_L"] == pairs["AddressLine1_clean_R"])
].copy()

r2d["MATCH_RULE"] = "NAME_DOB_ADDRESS"
r2d["MATCH_CONFIDENCE"] = 0.97
rules.append(r2d)

# ---------------------------------------------------------
# RULE 2E: NAME + DOB + SEX
# ---------------------------------------------------------

r2e = pairs[
    (pairs["FirstNM_clean_L"] == pairs["FirstNM_clean_R"]) &
    (pairs["LastNM_clean_L"] == pairs["LastNM_clean_R"]) &
    (pairs["BirthDT_clean_L"] == pairs["BirthDT_clean_R"]) &
    (pairs["SexAtBirthDSC_clean_L"] == pairs["SexAtBirthDSC_clean_R"])
].copy()

r2e["MATCH_RULE"] = "NAME_DOB_SEX"
r2e["MATCH_CONFIDENCE"] = 0.98
rules.append(r2e)

# ---------------------------------------------------------
# RULE 4: EMAIL EXACT
# ---------------------------------------------------------

r4 = pairs[
    pairs["Email_clean_L"].notna() &
    (pairs["Email_clean_L"] == pairs["Email_clean_R"])
].copy()

r4["MATCH_RULE"] = "EMAIL_EXACT"
r4["MATCH_CONFIDENCE"] = 0.995
rules.append(r4)


# =========================================================
# FINAL OUTPUT
# =========================================================

all_matches = pd.concat(rules, ignore_index=True)

all_matches = (
    all_matches
    .sort_values("MATCH_CONFIDENCE", ascending=False)
    .drop_duplicates(subset=["PATID_L", "PATID_R"])
)

final_matches = all_matches[
    ["PATID_L", "PATID_R", "MATCH_RULE", "MATCH_CONFIDENCE"]
]

print(final_matches.head())
print("\nTotal Matches:", len(final_matches))
print("\nRule Distribution:")
print(final_matches["MATCH_RULE"].value_counts())

                                PATID_L                           PATID_R  \
0      BF7DDA0309D9F928CF0B85D2E133F8D9  D085AC53A8D661D46E3951A994AC7223   
17948  5819D55392D56DA7C40194774DBAF442  7AB8B9FF9D52BA06FFF8B4AE2E6727F8   
17920  2D00CABDA743A91F7A634FDD4C4838DF  D7ADE59E979363BDE6C80D3A85F530DC   
17919  992D708B136929EEF3A3B7AC2DD9B976  ACBE4012BE4DCD379B769F9D32C5AC0D   
17918  00C53229C4BCC4671F9FAFAF51D048F1  C58438CFB97C02A52DEC2045DA53495A   

      MATCH_RULE  MATCH_CONFIDENCE  
0      EXACT_SSN               1.0  
17948  EXACT_SSN               1.0  
17920  EXACT_SSN               1.0  
17919  EXACT_SSN               1.0  
17918  EXACT_SSN               1.0  

Total Matches: 46669

Rule Distribution:
MATCH_RULE
NAME_DOB_SEX        18094
EMAIL_EXACT         12599
NAME_DOB_PHONE       8723
EXACT_SSN            5695
NAME_DOB_ADDRESS     1558
Name: count, dtype: int64


In [98]:
len(set(final_matches["PATID_L"].unique()) | set(final_matches["PATID_R"].unique())) / len(df)

0.33605935212164245

In [99]:
final_matches.value_counts("MATCH_RULE")

MATCH_RULE
NAME_DOB_SEX        18094
EMAIL_EXACT         12599
NAME_DOB_PHONE       8723
EXACT_SSN            5695
NAME_DOB_ADDRESS     1558
Name: count, dtype: int64

In [100]:
exact_ssn_pairs = final_matches.copy()

exact_ssn_review = exact_ssn_pairs.merge(
    df.add_suffix("_L"),
    left_on="PATID_L",
    right_on="PATID_L",
    how="left"
).merge(
    df.add_suffix("_R"),
    left_on="PATID_R",
    right_on="PATID_R",
    how="left"
)

review_cols = [
    "PATID_L",
    "PATID_R",
    "MATCH_RULE",
    "MATCH_CONFIDENCE",

    "FirstNM_clean_L",
    "FirstNM_clean_R",

    "LastNM_clean_L",
    "LastNM_clean_R",

    "BirthDT_clean_L",
    "BirthDT_clean_R",

    "SSN_clean_L",
    "SSN_clean_R",

    "PhonePrimary_clean_L",
    "PhonePrimary_clean_R",

    "AddressLine1_clean_L",
    "AddressLine1_clean_R",

    "Email_clean_L",
    "Email_clean_R"
]

exact_ssn_review = exact_ssn_review[review_cols]

print(exact_ssn_review.head())

                            PATID_L                           PATID_R  \
0  BF7DDA0309D9F928CF0B85D2E133F8D9  D085AC53A8D661D46E3951A994AC7223   
1  5819D55392D56DA7C40194774DBAF442  7AB8B9FF9D52BA06FFF8B4AE2E6727F8   
2  2D00CABDA743A91F7A634FDD4C4838DF  D7ADE59E979363BDE6C80D3A85F530DC   
3  992D708B136929EEF3A3B7AC2DD9B976  ACBE4012BE4DCD379B769F9D32C5AC0D   
4  00C53229C4BCC4671F9FAFAF51D048F1  C58438CFB97C02A52DEC2045DA53495A   

  MATCH_RULE  MATCH_CONFIDENCE FirstNM_clean_L FirstNM_clean_R LastNM_clean_L  \
0  EXACT_SSN               1.0       RONZJANEL       RONZJANEL       WOOLFOLK   
1  EXACT_SSN               1.0           LAMAR           LAMAR      DONALDSON   
2  EXACT_SSN               1.0           BRYNN           BRYNN            LAW   
3  EXACT_SSN               1.0           MARIA           MARIA     VILLARREAL   
4  EXACT_SSN               1.0         KENNETH         KENNETH         ERVING   

  LastNM_clean_R BirthDT_clean_L BirthDT_clean_R  SSN_clean_L  SSN_clean_R

In [101]:
exact_ssn_review.to_csv('../data/review/deterministic_matches.csv', index=False)

In [102]:
import pandas as pd
import numpy as np

# =========================================================
# FULL RULE EVALUATION FOR ALL MATCH TYPES
# =========================================================
#
# PURPOSE
# -------
# Evaluate deterministic EMPI linkage quality
# across ALL match rules.
#
# INPUTS
# -------
# final_matches
# df
#
# OUTPUTS
# -------
# 1. Rule-level agreement metrics
# 2. Contradiction analysis
# 3. Suspicious match detection
# 4. Cluster analysis
# 5. Rule quality ranking
#
# =========================================================

# =========================================================
# ATTACH DEMOGRAPHICS
# =========================================================

review_df = final_matches.merge(
    df.add_suffix("_L"),
    left_on="PATID_L",
    right_on="PATID_L",
    how="left"
).merge(
    df.add_suffix("_R"),
    left_on="PATID_R",
    right_on="PATID_R",
    how="left"
)

# =========================================================
# REVIEW COLUMNS
# =========================================================

review_cols = [
    "PATID_L",
    "PATID_R",

    "MATCH_RULE",
    "MATCH_CONFIDENCE",

    "FirstNM_clean_L",
    "FirstNM_clean_R",

    "LastNM_clean_L",
    "LastNM_clean_R",

    "BirthDT_clean_L",
    "BirthDT_clean_R",

    "SexAtBirthDSC_clean_L",
    "SexAtBirthDSC_clean_R",

    "SSN_clean_L",
    "SSN_clean_R",

    "PhonePrimary_clean_L",
    "PhonePrimary_clean_R",

    "AddressLine1_clean_L",
    "AddressLine1_clean_R",

    "Email_clean_L",
    "Email_clean_R"
]

review_df = review_df[review_cols]

# =========================================================
# RULE-LEVEL METRICS
# =========================================================

rule_results = []

for rule in sorted(review_df["MATCH_RULE"].unique()):

    subset = review_df[
        review_df["MATCH_RULE"] == rule
    ].copy()

    n_matches = len(subset)

    # -----------------------------------------------------
    # AGREEMENT METRICS
    # -----------------------------------------------------

    first_agree = (
        subset["FirstNM_clean_L"] ==
        subset["FirstNM_clean_R"]
    ).mean()

    last_agree = (
        subset["LastNM_clean_L"] ==
        subset["LastNM_clean_R"]
    ).mean()

    dob_agree = (
        subset["BirthDT_clean_L"] ==
        subset["BirthDT_clean_R"]
    ).mean()

    sex_agree = (
        subset["SexAtBirthDSC_clean_L"] ==
        subset["SexAtBirthDSC_clean_R"]
    ).mean()

    ssn_agree = (
        subset["SSN_clean_L"] ==
        subset["SSN_clean_R"]
    ).mean()

    phone_agree = (
        subset["PhonePrimary_clean_L"] ==
        subset["PhonePrimary_clean_R"]
    ).mean()

    address_agree = (
        subset["AddressLine1_clean_L"] ==
        subset["AddressLine1_clean_R"]
    ).mean()

    email_agree = (
        subset["Email_clean_L"] ==
        subset["Email_clean_R"]
    ).mean()

    # -----------------------------------------------------
    # CONTRADICTION METRICS
    # -----------------------------------------------------

    dob_mismatch = (
        subset["BirthDT_clean_L"] !=
        subset["BirthDT_clean_R"]
    ).mean()

    sex_mismatch = (
        subset["SexAtBirthDSC_clean_L"] !=
        subset["SexAtBirthDSC_clean_R"]
    ).mean()

    last_mismatch = (
        subset["LastNM_clean_L"] !=
        subset["LastNM_clean_R"]
    ).mean()

    ssn_mismatch = (
        (
            subset["SSN_clean_L"].notna()
        ) &
        (
            subset["SSN_clean_R"].notna()
        ) &
        (
            subset["SSN_clean_L"] !=
            subset["SSN_clean_R"]
        )
    ).mean()

    # -----------------------------------------------------
    # QUALITY SCORE
    # -----------------------------------------------------
    #
    # Composite score based on demographic agreement.
    #
    # Higher = better rule quality
    #
    # -----------------------------------------------------

    quality_score = np.mean([
        first_agree,
        last_agree,
        dob_agree,
        sex_agree
    ])

    # -----------------------------------------------------
    # STORE RESULTS
    # -----------------------------------------------------

    rule_results.append({
        "MATCH_RULE": rule,
        "N_MATCHES": n_matches,

        "FIRSTNAME_AGREEMENT":
            round(first_agree, 4),

        "LASTNAME_AGREEMENT":
            round(last_agree, 4),

        "DOB_AGREEMENT":
            round(dob_agree, 4),

        "SEX_AGREEMENT":
            round(sex_agree, 4),

        "SSN_AGREEMENT":
            round(ssn_agree, 4),

        "PHONE_AGREEMENT":
            round(phone_agree, 4),

        "ADDRESS_AGREEMENT":
            round(address_agree, 4),

        "EMAIL_AGREEMENT":
            round(email_agree, 4),

        "DOB_MISMATCH":
            round(dob_mismatch, 4),

        "SEX_MISMATCH":
            round(sex_mismatch, 4),

        "LASTNAME_MISMATCH":
            round(last_mismatch, 4),

        "SSN_MISMATCH":
            round(ssn_mismatch, 4),

        "QUALITY_SCORE":
            round(quality_score, 4)
    })

# =========================================================
# FINAL RULE METRICS TABLE
# =========================================================

rule_metrics_df = pd.DataFrame(rule_results)

rule_metrics_df = rule_metrics_df.sort_values(
    "QUALITY_SCORE",
    ascending=False
)

print("\n====================================================")
print("RULE QUALITY METRICS")
print("====================================================")

print(rule_metrics_df)

# =========================================================
# INTERPRETATION GUIDE
# =========================================================
#
# QUALITY SCORE
# -------------
# > 0.95 = excellent deterministic rule
# 0.85-0.95 = strong rule
# 0.70-0.85 = moderate rule
# < 0.70 = risky / noisy rule
#
# IMPORTANT METRICS
# -----------------
# DOB_MISMATCH:
#     should usually be near 0
#
# SSN_MISMATCH:
#     should be VERY low
#
# LASTNAME_MISMATCH:
#     high values indicate weak rule
#
# =========================================================

# =========================================================
# SUSPICIOUS MATCHES
# =========================================================
#
# PURPOSE:
# Surface likely false positives.
#
# ---------------------------------------------------------

suspicious_matches = review_df[
    (
        review_df["BirthDT_clean_L"] !=
        review_df["BirthDT_clean_R"]
    ) |
    (
        review_df["LastNM_clean_L"] !=
        review_df["LastNM_clean_R"]
    ) |
    (
        (
            review_df["SSN_clean_L"].notna()
        ) &
        (
            review_df["SSN_clean_R"].notna()
        ) &
        (
            review_df["SSN_clean_L"] !=
            review_df["SSN_clean_R"]
        )
    )
]

print("\n====================================================")
print("SUSPICIOUS MATCHES")
print("====================================================")

print(f"Total suspicious matches: {len(suspicious_matches)}")

print(
    suspicious_matches.head(25)
)

# =========================================================
# CLUSTER ANALYSIS
# =========================================================
#
# PURPOSE:
# Detect over-linked patients.
#
# Large clusters often indicate:
# - bad blocking
# - common placeholders
# - overly broad rules
#
# ---------------------------------------------------------

left_counts = review_df["PATID_L"].value_counts()
right_counts = review_df["PATID_R"].value_counts()

cluster_counts = (
    left_counts.add(right_counts, fill_value=0)
)

cluster_df = cluster_counts.reset_index()
cluster_df.columns = ["PATID", "LINK_COUNT"]

print("\n====================================================")
print("LARGEST MATCH CLUSTERS")
print("====================================================")

print(
    cluster_df.sort_values(
        "LINK_COUNT",
        ascending=False
    ).head(20)
)

# =========================================================
# RULE DISTRIBUTION
# =========================================================

rule_distribution = (
    review_df["MATCH_RULE"]
    .value_counts()
    .reset_index()
)

rule_distribution.columns = [
    "MATCH_RULE",
    "COUNT"
]

print("\n====================================================")
print("RULE DISTRIBUTION")
print("====================================================")

print(rule_distribution)

# =========================================================
# OVERALL SUMMARY
# =========================================================

summary = {
    "TOTAL_MATCHES":
        len(review_df),

    "TOTAL_RULES":
        review_df["MATCH_RULE"].nunique(),

    "AVG_MATCH_CONFIDENCE":
        round(
            review_df["MATCH_CONFIDENCE"].mean(),
            4
        ),

    "SUSPICIOUS_MATCH_RATE":
        round(
            len(suspicious_matches) /
            max(len(review_df), 1),
            4
        ),

    "MAX_CLUSTER_SIZE":
        int(cluster_df["LINK_COUNT"].max())
}

summary_df = pd.DataFrame(
    summary.items(),
    columns=["METRIC", "VALUE"]
)

print("\n====================================================")
print("OVERALL SUMMARY")
print("====================================================")

print(summary_df)


RULE QUALITY METRICS
         MATCH_RULE  N_MATCHES  FIRSTNAME_AGREEMENT  LASTNAME_AGREEMENT  \
4      NAME_DOB_SEX      18094               1.0000              1.0000   
3    NAME_DOB_PHONE       8723               1.0000              1.0000   
1         EXACT_SSN       5695               0.8342              0.7468   
2  NAME_DOB_ADDRESS       1558               1.0000              1.0000   
0       EMAIL_EXACT      12599               0.2826              0.2826   

   DOB_AGREEMENT  SEX_AGREEMENT  SSN_AGREEMENT  PHONE_AGREEMENT  \
4         1.0000         1.0000            0.0           0.0000   
3         1.0000         0.7252            0.0           1.0000   
1         0.8739         0.7751            1.0           0.2992   
2         1.0000         0.0000            0.0           0.0000   
0         0.3278         0.5526            0.0           0.2050   

   ADDRESS_AGREEMENT  EMAIL_AGREEMENT  DOB_MISMATCH  SEX_MISMATCH  \
4             0.1057           0.0000        0.0000    

In [103]:
# =========================================================
# PATIENT COVERAGE ANALYSIS
# =========================================================
#
# PURPOSE
# -------
# Measure how much of the population was covered
# by deterministic linkage.
#
# COVERAGE METRICS
# ----------------
# 1. Total patients
# 2. Patients involved in ANY match
# 3. Patients NOT matched
# 4. Coverage rate
# 5. Coverage by rule
# 6. Multi-match patients
#
# INPUTS
# -------
# df
# final_matches
#
# =========================================================

import pandas as pd
import numpy as np

# =========================================================
# TOTAL UNIQUE PATIENTS
# =========================================================

total_patients = df["PATID"].nunique()

print("\n====================================================")
print("TOTAL PATIENT POPULATION")
print("====================================================")

print(f"Total unique patients: {total_patients:,}")

# =========================================================
# PATIENTS COVERED BY MATCHES
# =========================================================

matched_patients = pd.unique(
    pd.concat([
        final_matches["PATID_L"],
        final_matches["PATID_R"]
    ])
)

matched_patients = pd.Series(matched_patients)

n_matched_patients = matched_patients.nunique()

coverage_rate = n_matched_patients / total_patients

# =========================================================
# UNMATCHED PATIENTS
# =========================================================

all_patients = set(df["PATID"].unique())

matched_patient_set = set(matched_patients.unique())

unmatched_patients = all_patients - matched_patient_set

n_unmatched = len(unmatched_patients)

# =========================================================
# SUMMARY
# =========================================================

coverage_summary = pd.DataFrame({
    "Metric": [
        "Total Patients",
        "Patients in ≥1 Match",
        "Unmatched Patients",
        "Coverage Rate"
    ],
    "Value": [
        total_patients,
        n_matched_patients,
        n_unmatched,
        round(coverage_rate, 4)
    ]
})

print("\n====================================================")
print("OVERALL COVERAGE SUMMARY")
print("====================================================")

print(coverage_summary)

# =========================================================
# COVERAGE BY MATCH RULE
# =========================================================
#
# PURPOSE:
# Determine how many UNIQUE patients
# each rule accounts for.
#
# ---------------------------------------------------------

rule_coverage = []

for rule in sorted(final_matches["MATCH_RULE"].unique()):

    subset = final_matches[
        final_matches["MATCH_RULE"] == rule
    ]

    patients_in_rule = pd.unique(
        pd.concat([
            subset["PATID_L"],
            subset["PATID_R"]
        ])
    )

    n_patients = len(patients_in_rule)

    pct_population = n_patients / total_patients

    rule_coverage.append({
        "MATCH_RULE": rule,
        "UNIQUE_PATIENTS_COVERED": n_patients,
        "POPULATION_COVERAGE":
            round(pct_population, 4),
        "MATCH_COUNT":
            len(subset)
    })

rule_coverage_df = pd.DataFrame(rule_coverage)

rule_coverage_df = rule_coverage_df.sort_values(
    "UNIQUE_PATIENTS_COVERED",
    ascending=False
)

print("\n====================================================")
print("COVERAGE BY MATCH RULE")
print("====================================================")

print(rule_coverage_df)

# =========================================================
# MULTI-MATCH ANALYSIS
# =========================================================
#
# PURPOSE:
# Detect patients linked to many others.
#
# Large numbers may indicate:
# - duplicate-heavy records
# - overly broad rules
# - placeholder demographics
#
# ---------------------------------------------------------

left_counts = (
    final_matches["PATID_L"]
    .value_counts()
)

right_counts = (
    final_matches["PATID_R"]
    .value_counts()
)

combined_counts = (
    left_counts.add(right_counts, fill_value=0)
)

combined_counts = combined_counts.reset_index()

combined_counts.columns = [
    "PATID",
    "N_LINKS"
]

print("\n====================================================")
print("PATIENTS WITH MOST LINKS")
print("====================================================")

print(
    combined_counts.sort_values(
        "N_LINKS",
        ascending=False
    ).head(20)
)

# =========================================================
# MATCH DISTRIBUTION
# =========================================================
#
# PURPOSE:
# Understand linkage density.
#
# ---------------------------------------------------------

distribution = (
    combined_counts["N_LINKS"]
    .value_counts()
    .sort_index()
)

distribution_df = distribution.reset_index()

distribution_df.columns = [
    "NUMBER_OF_LINKS",
    "NUMBER_OF_PATIENTS"
]

print("\n====================================================")
print("LINK DISTRIBUTION")
print("====================================================")

print(distribution_df.head(20))

# =========================================================
# OPTIONAL:
# SHOW UNMATCHED SAMPLE
# =========================================================

unmatched_sample = (
    df[df["PATID"].isin(unmatched_patients)]
    .head(20)
)

print("\n====================================================")
print("SAMPLE UNMATCHED PATIENTS")
print("====================================================")

print(
    unmatched_sample[
        [
            "PATID",
            "FirstNM_clean",
            "LastNM_clean",
            "BirthDT_clean"
        ]
    ]
)

# =========================================================
# INTERPRETATION GUIDE
# =========================================================
#
# GOOD SIGNS
# ----------
# - Moderate coverage (depends on duplicate prevalence)
# - Small cluster sizes
# - Most patients have 1-3 links max
#
# WARNING SIGNS
# -------------
# - Extremely high coverage (>50% unexpectedly)
# - Huge clusters
# - One rule dominating all matches
#
# EXPECTED COVERAGE
# -----------------
# Typical EMPI duplicate rates:
#
# 2% - 15% of patients
#
# depending on:
# - source systems
# - ingestion quality
# - historical merges
#
# =========================================================


TOTAL PATIENT POPULATION
Total unique patients: 163,364

OVERALL COVERAGE SUMMARY
                 Metric        Value
0        Total Patients  163364.0000
1  Patients in ≥1 Match   54900.0000
2    Unmatched Patients  108464.0000
3         Coverage Rate       0.3361

COVERAGE BY MATCH RULE
         MATCH_RULE  UNIQUE_PATIENTS_COVERED  POPULATION_COVERAGE  MATCH_COUNT
4      NAME_DOB_SEX                    27100               0.1659        18094
3    NAME_DOB_PHONE                    15143               0.0927         8723
0       EMAIL_EXACT                     8830               0.0541        12599
1         EXACT_SSN                     8313               0.0509         5695
2  NAME_DOB_ADDRESS                     2953               0.0181         1558

PATIENTS WITH MOST LINKS
                                  PATID  N_LINKS
54764  FF69B2FE0378A362F6B6FEA3D2C03FA4     91.0
45019  D25D9265E706CC44A7F4D49D4EADE0DF     91.0
52719  F5C4571B322CD3D3193574397FFDA56E     91.0
40806  BEF60